# 11. Проект. Модель для показа рекламных предложений

В этом проекте вам необходимо реализовать мониторинг модели с помощью Prometheus и Grafana. Вы можете сами выбрать решаемую задачу и датасет. Если не знаете, что выбрать, то можете остановиться на этих данных "winequality-red.csv"

Этапы, которые нужно пройти:
- Создание модели.
- Скраппинг метрик модели с помощью Prometheus.
- Подключение Prometheus к Grafana.
- Отрисовка дашборда.

С нуля писать взаимодействие между всеми этими компонентами довольно сложно. Будет очень здорово, если у вас получится отобразить хотя бы score модели на дашборде Grafana, который будет передаваться через Prometheus. 

Если же вы хотите более красочный дашборд, то можно пойти немного другим путём и использовать уже заранее созданные взаимодействия между этими компонентами в Kubernetes. Подробная инструкция https://github.com/PsychoBel/Monitoring-ML-models

### Критерии проверки
Критерий
- Создана модель - 2 балла
- Метрики модели передаются в Prometheus - 2 балла
- На дашборде отображены метрики модели - 4 балла
- Модель настроена так, что её можно дообучать - 2 балла

### Импорт библиотек

In [7]:
import numpy as np
import pandas as pd
import plotly.express as px
import lightgbm as lgb
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
from sklearn.metrics import f1_score, precision_score, recall_score

from skopt import BayesSearchCV
from skopt.space import Real, Categorical

### Загрузка датасета

In [8]:
dataset = pd.read_csv('data/winequality-red.csv', sep=';')
dataset.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


## Анализ данных был выполнен ранее в разделе Kernell Function (Ядерные функции). Буду использовать уже готовое решение с Байесовской оптимизацией из домашнего задания "DS-ADD-4".

In [13]:
# Реализация стримингового прочтение файлов:
def streaming_reading(file_path, sep=';', batch_size=5000):
    for chunk in pd.read_csv(file_path, sep=sep, chunksize=batch_size):
        chunk = chunk.copy()
        chunk['good_wine'] = (chunk['quality'] >= 6.5).astype(int)
        chunk = chunk.drop(columns=['quality'])
        chunk['so2_ratio'] = chunk['free sulfur dioxide'] / (chunk['total sulfur dioxide'] + 1e-6)
        X_chunk = chunk.drop(columns=['good_wine'])
        y_chunk = chunk['good_wine']
        yield X_chunk, y_chunk

X_parts = []
y_parts = []
for X_chunk, y_chunk in streaming_reading('data/winequality-red.csv', batch_size=5000):
    X_parts.append(X_chunk)
    y_parts.append(y_chunk)

X = pd.concat(X_parts, ignore_index=True)
y = pd.concat(y_parts, ignore_index=True)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_model = pd.DataFrame(X_scaled, columns=X.columns)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_model,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# Bayesian optimization for SVC
search_spaces = {
    'C': Real(1e-2, 1e2, prior='log-uniform'),
    'gamma': Real(1e-3, 1e1, prior='log-uniform'),
    'kernel': Categorical(['linear', 'poly', 'rbf', 'sigmoid'])
}

bayes_search = BayesSearchCV(
    estimator=SVC(),
    search_spaces=search_spaces,
    n_iter=40,
    scoring='accuracy',
    cv=5,
    n_jobs=-1,
    random_state=42,
)

bayes_search.fit(X_train, y_train)
best_svc = bayes_search.best_estimator_

valid_pred = best_svc.predict(X_valid)
valid_accuracy = accuracy_score(y_valid, valid_pred)
print('Best params:', bayes_search.best_params_)
print(f'Best CV accuracy (BayesSearchCV): {bayes_search.best_score_:.4f}')
print(f'Validation accuracy: {valid_accuracy:.4f}')

cm = confusion_matrix(y_valid, valid_pred)
cm_df = pd.DataFrame(
    cm,
    index=['actual_bad(0)', 'actual_good(1)'],
    columns=['pred_bad(0)', 'pred_good(1)']
)
print('Confusion matrix:')
display(cm_df)

cv_scores = cross_val_score(best_svc, X_model, y, cv=5, scoring='accuracy')
print('Cross-val accuracy scores:', np.round(cv_scores, 4))
print(f'Cross-val mean accuracy: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}')

# Compare several models
models = {
    'LogisticRegression': LogisticRegression(max_iter=3000, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=7),
    'RandomForest': RandomForestClassifier(n_estimators=300, random_state=42),
    'SVC_best': best_svc,
}

metrics_rows = []
for model_name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_valid)
    metrics_rows.append({
        'model': model_name,
        'val_accuracy': accuracy_score(y_valid, pred),
        'val_precision': precision_score(y_valid, pred, zero_division=0),
        'val_recall': recall_score(y_valid, pred, zero_division=0),
        'val_f1': f1_score(y_valid, pred, zero_division=0),
        'cv_accuracy_mean': cross_val_score(model, X_model, y, cv=5, scoring='accuracy').mean(),
        'cv_accuracy_std': cross_val_score(model, X_model, y, cv=5, scoring='accuracy').std(),
    })

comparison_df = pd.DataFrame(metrics_rows).sort_values('cv_accuracy_mean', ascending=False).reset_index(drop=True)
print('Сравнение метрик моделей:')
display(comparison_df.round(4))

# Save the best model for monitoring/deployment
joblib.dump(best_svc, 'model.joblib')
print('Best model saved to model.joblib')

# Plot comparison: accuracy on validation and CV
plot_df = comparison_df.melt(
    id_vars='model',
    value_vars=['val_accuracy', 'cv_accuracy_mean'],
    var_name='metric',
    value_name='score'
)
fig_cmp_acc = px.bar(
    plot_df,
    x='model',
    y='score',
    color='metric',
    barmode='group',
    text='score',
    title='Сравнение accuracy по моделям'
)
fig_cmp_acc.update_traces(texttemplate='%{text:.3f}', textposition='outside')
fig_cmp_acc.update_layout(yaxis_range=[0.7, 1.0])
fig_cmp_acc.show()

# Plot comparison: precision/recall/F1
plot_df_prf = comparison_df.melt(
    id_vars='model',
    value_vars=['val_precision', 'val_recall', 'val_f1'],
    var_name='metric',
    value_name='score'
)
fig_cmp_prf = px.bar(
    plot_df_prf,
    x='model',
    y='score',
    color='metric',
    barmode='group',
    text='score',
    title='Сравнение precision / recall / F1 (validation)'
)
fig_cmp_prf.update_traces(texttemplate='%{text:.3f}', textposition='outside')
fig_cmp_prf.update_layout(yaxis_range=[0.0, 1.0])
fig_cmp_prf.show()


Best params: OrderedDict([('C', 99.93242189807779), ('gamma', 0.6084342505025045), ('kernel', 'rbf')])
Best CV accuracy (BayesSearchCV): 0.9062
Validation accuracy: 0.9219
Confusion matrix:


,pred_bad(0),pred_good(1)
actual_bad(0),272,5
actual_good(1),20,23


Cross-val accuracy scores: [0.8688 0.8562 0.8562 0.8188 0.8683]
Cross-val mean accuracy: 0.8537 +/- 0.0183
Сравнение метрик моделей:


,model,val_accuracy,val_precision,val_recall,val_f1,cv_accuracy_mean,cv_accuracy_std
0,RandomForest,0.9406,0.9286,0.6047,0.7324,0.8724,0.0138
1,LogisticRegression,0.8906,0.6818,0.3488,0.4615,0.8656,0.0231
2,KNN,0.9000,0.6774,0.4884,0.5676,0.8562,0.0355
3,SVC_best,0.9219,0.8214,0.5349,0.6479,0.8537,0.0183


Best model saved to model.joblib
